# Music genre classification

## 1. Data & Setup

In [ ]:
from pathlib import Path
import os, json, hashlib, inspect, platform, time
os.environ['OMP_NUM_THREADS'] = '1'
os.environ['OPENBLAS_NUM_THREADS'] = '1'
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy, sklearn
from scipy.stats import skew
from sklearn.base import clone
from sklearn.model_selection import train_test_split, StratifiedKFold, KFold, GridSearchCV
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler, label_binarize
from sklearn.dummy import DummyClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import BaggingClassifier, RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import (accuracy_score, f1_score, precision_recall_fscore_support,
    confusion_matrix, ConfusionMatrixDisplay, roc_auc_score, roc_curve,
    precision_recall_curve, average_precision_score, auc, recall_score)
from threadpoolctl import threadpool_limits
from IPython.display import display
threadpool_limits(1)
SEED = 42
OUT = Path('hw2_results'); OUT.mkdir(exist_ok=True)
FEATURES = ['danceability','energy','speechiness','acousticness','instrumentalness',
            'liveness','valence','loudness','tempo','key','mode','time_signature','duration_ms']
RARE = ['death-metal','tango','opera','black-metal','salsa']
df = pd.read_csv('processed_tracks.csv')
assert df.shape == (5300,20) and df.track_id.is_unique
assert not df[FEATURES].isna().any().any()
dev, test = train_test_split(np.arange(len(df)), test_size=.2,
                            stratify=df.track_genre, random_state=SEED)
if Path('preregistration_support/development_rows.csv').exists():
    assert np.array_equal(dev, pd.read_csv('preregistration_support/development_rows.csv').original_row)
X = df.iloc[dev][FEATURES].to_numpy(); y = df.iloc[dev].track_genre.to_numpy()
LABELS = np.unique(y)
outer = list(StratifiedKFold(5, shuffle=True, random_state=SEED).split(X,y))
def inner_cv(): return StratifiedKFold(3, shuffle=True, random_state=SEED)
def table(name, data):
    d = data if isinstance(data,pd.DataFrame) else pd.DataFrame(data)
    d.to_csv(OUT/(name+'.csv'), index=False); display(d); return d
def savefig(name):
    plt.tight_layout(); plt.savefig(OUT/(name+'.png'),dpi=160,bbox_inches='tight'); plt.show(); plt.close()
def scores(a,b):
    return {'accuracy':accuracy_score(a,b), 'macro_f1':f1_score(a,b,labels=LABELS,average='macro',zero_division=0)}
def class_table(a,b):
    p,r,f,s = precision_recall_fscore_support(a,b,labels=LABELS,zero_division=0)
    return pd.DataFrame(dict(genre=LABELS,precision=p,recall=r,f1=f,support=s))
def evaluate(model, name, xx=X, yy=y, save_predictions=True):
    rows=[]; pred=np.empty_like(yy); prob=None
    for fold,(t,v) in enumerate(outer):
        m=clone(model).fit(xx[t],yy[t]); p=m.predict(xx[v]); pred[v]=p
        rows.append(dict(model=name,fold=fold+1,**scores(yy[v],p),
                         train_accuracy=accuracy_score(yy[t],m.predict(xx[t])),
                         train_macro_f1=f1_score(yy[t],m.predict(xx[t]),labels=LABELS,average='macro',zero_division=0),
                         leaves=m.get_n_leaves() if hasattr(m,'get_n_leaves') else np.nan))
        if hasattr(m,'predict_proba'):
            if prob is None: prob=np.zeros((len(yy),len(LABELS)))
            prob[v]=m.predict_proba(xx[v])
    if save_predictions:
        pd.DataFrame(dict(original_row=dev,truth=yy,prediction=pred)).to_csv(OUT/(name+'_oof.csv'),index=False)
    return pd.DataFrame(rows),pred,prob
def curve_plot(d,param,name,log=False):
    fig,axs=plt.subplots(1,2,figsize=(11,4))
    for ax,metric in zip(axs,['accuracy','macro_f1']):
        for col,label in [(metric,'validation'),('train_'+metric,'training')]:
            g=d.groupby(param)[col].agg(['mean','std']).sort_index()
            ax.errorbar(g.index,g['mean'],yerr=g['std'],label=label,marker='.',capsize=2)
        ax.set(xlabel=param,ylabel=metric); ax.legend()
        if log: ax.set_xscale('log')
    savefig(name)
table('data_counts',df.track_genre.value_counts().rename_axis('genre').reset_index(name='count'))
manifest=dict(seed=SEED,test_fraction=.2,outer_folds=5,inner_folds=3,
    data_sha256=hashlib.sha256(Path('processed_tracks.csv').read_bytes()).hexdigest(),
    versions=dict(python=platform.python_version(),numpy=np.__version__,pandas=pd.__version__,
                  scipy=scipy.__version__,sklearn=sklearn.__version__))
(OUT/'manifest.json').write_text(json.dumps(manifest,indent=2))
table('split_membership',pd.DataFrame(dict(original_row=np.r_[dev,test],partition=['development']*len(dev)+['test']*len(test))))

## 2. Protocols

In [ ]:
rows=[]
for regime,cv in [('random',KFold(5,shuffle=True,random_state=SEED)),
                  ('stratified',StratifiedKFold(5,shuffle=True,random_state=SEED))]:
    for f,(_,v) in enumerate(cv.split(X,y)):
        for g in RARE: rows.append(dict(regime=regime,fold=f+1,genre=g,count=int(sum(y[v]==g))))
table('rare_fold_counts',rows)
base=[]
for strategy in ['most_frequent','stratified']:
    d,p,_=evaluate(DummyClassifier(strategy=strategy,random_state=SEED),strategy)
    d['micro_f1']=d.accuracy
    assert np.isclose(f1_score(y,p,average='micro'),accuracy_score(y,p))
    base.append(d)
table('baseline_scores',pd.concat(base))
prior=pd.Series(y).value_counts(normalize=True)
table('baseline_formula_values',[dict(majority_accuracy=prior.max(),
    majority_macro_f1=2*prior.max()/(1+prior.max())/20,
    random_expected_accuracy=sum(prior**2),random_large_sample_macro_f1=1/20)])

## 3.1 k-NN

In [ ]:
KS=[1,3,5,7,9,15,21,31,51,75,100,125]
knn_rows=[]; rare_rows=[]
for k in KS:
    d,p,_=evaluate(make_pipeline(StandardScaler(),KNeighborsClassifier(k,n_jobs=1)),f'knn_{k}',save_predictions=False)
    d['k']=k; knn_rows.append(d)
    c=class_table(y,p).set_index('genre').loc[RARE]
    for g in RARE: rare_rows.append(dict(k=k,genre=g,recall=c.loc[g,'recall']))
    rare_rows.append(dict(k=k,genre='rare_average',recall=c.recall.mean()))
knn_curves=table('knn_curves',pd.concat(knn_rows)); table('knn_rare_recall',rare_rows)
curve_plot(knn_curves,'k','knn_complexity',True)
print('Best development k by each metric:',knn_curves.groupby('k')[['accuracy','macro_f1']].mean().idxmax().to_dict())
t,v=outer[0]; scaler=StandardScaler().fit(X[t]); zt=scaler.transform(X[t]); zv=scaler.transform(X[v])
demo_knn=KNeighborsClassifier(5).fit(zt,y[t]); pp=demo_knn.predict(zv)
wrong=np.flatnonzero(pp!=y[v]); rare_wrong=[i for i in wrong if y[v[i]] in RARE]
assert rare_wrong
chosen=list(dict.fromkeys([rare_wrong[0]]+list(wrong)))[:3]
dist,idx=demo_knn.kneighbors(zv[chosen]); rows=[]
for j,i in enumerate(chosen):
    votes=pd.Series(y[t][idx[j]]).value_counts().to_dict()
    for rank,(dd,ii) in enumerate(zip(dist[j],idx[j])):
        rows.append(dict(validation_row=int(dev[v[i]]),truth=y[v[i]],prediction=pp[i],
            rank=rank+1,neighbor_row=int(dev[t[ii]]),neighbor_genre=y[t[ii]],distance=dd,votes=json.dumps(votes)))
table('knn_wrong_neighbors',rows)
noise=np.random.default_rng(SEED).normal(size=(len(df),50))
noise_rows=[]; retain=[]
for n in [0,5,20,50]:
    pred=np.empty_like(y)
    for f,(tt,vv) in enumerate(outer):
        sc=StandardScaler().fit(X[tt]); a=sc.transform(X[tt]); b=sc.transform(X[vv])
        clean=KNeighborsClassifier(5).fit(a,y[tt]); old=clean.kneighbors(b,return_distance=False)
        noisy=KNeighborsClassifier(5).fit(np.c_[a,noise[dev[tt],:n]],y[tt])
        bn=np.c_[b,noise[dev[vv],:n]]; pred[vv]=noisy.predict(bn)
        new=noisy.kneighbors(bn,return_distance=False)
        for j,row in enumerate(vv): retain.append(dict(noise_columns=n,original_row=int(dev[row]),
            retained_fraction=len(set(old[j])&set(new[j]))/5))
        noise_rows.append(dict(noise_columns=n,fold=f+1,**scores(y[vv],pred[vv])))
table('knn_noise_scores',noise_rows); rr=table('knn_noise_retention_per_track',retain)
table('knn_noise_retention_summary',rr.groupby('noise_columns').retained_fraction.agg(['mean','std','min','max']).reset_index())
fig,ax=plt.subplots(); rr.boxplot(column='retained_fraction',by='noise_columns',ax=ax); plt.suptitle(''); savefig('knn_noise_retention')

## 3.2 Naive Bayes

In [ ]:
t,v=outer[0]; sc=StandardScaler().fit(X[t]); a=sc.transform(X[t]); b=sc.transform(X[v])
nb=GaussianNB().fit(a,y[t]); logp=nb.predict_log_proba(b)
i=0; aa,bb=np.argsort(logp[i])[-2:][::-1]; x=b[i]
va,vb=nb.var_[aa],nb.var_[bb]; ma,mb=nb.theta_[aa],nb.theta_[bb]
terms=-.5*np.log(va/vb)-.5*((x-ma)**2/va-(x-mb)**2/vb)
prior_term=np.log(nb.class_prior_[aa]/nb.class_prior_[bb])
assert np.isclose(prior_term+terms.sum(),logp[i,aa]-logp[i,bb])
table('nb_log_odds_terms',pd.DataFrame(dict(term=['prior']+FEATURES,value=np.r_[prior_term,terms],
    mean_A=np.r_[np.nan,ma],mean_B=np.r_[np.nan,mb],variance_A=np.r_[np.nan,va],variance_B=np.r_[np.nan,vb],
    x=np.r_[np.nan,x])))
print('Validation original row, truth, A, B:',int(dev[v[i]]),y[v[i]],nb.classes_[aa],nb.classes_[bb])
SMOOTH=[1e-12,1e-9,1e-7,1e-5,1e-3,1e-1,1.0]
nbrows=[]
for sm in SMOOTH:
    d,_,_=evaluate(make_pipeline(StandardScaler(),GaussianNB(var_smoothing=sm)),f'nb_{sm}',save_predictions=False)
    d['var_smoothing']=sm; nbrows.append(d)
nbcurve=table('nb_smoothing_curves',pd.concat(nbrows)); curve_plot(nbcurve,'var_smoothing','nb_smoothing',True)
NB_SMOOTH=float(nbcurve.groupby('var_smoothing').macro_f1.mean().idxmax())
raw_d,raw_p,_=evaluate(GaussianNB(),'nb_raw_default')
std_d,std_p,_=evaluate(make_pipeline(StandardScaler(),GaussianNB()),'nb_scaled_default')
table('nb_raw_scaled_scores',pd.concat([raw_d,std_d]))
raw=GaussianNB().fit(X[t],y[t]); scaled=GaussianNB().fit(a,y[t])
table('nb_scale_numerics',[dict(raw_epsilon=raw.epsilon_,scaled_epsilon=scaled.epsilon_,
    max_raw_variance=X[t].var(axis=0).max(),largest_variance_feature=FEATURES[X[t].var(axis=0).argmax()],
    raw_scaled_oof_disagreement=np.mean(raw_p!=std_p),chosen_smoothing=NB_SMOOTH)])
table('nb_default_class_scores',class_table(y,std_p))
varrows=[]
for j,g in enumerate(scaled.classes_):
    original_variance=a[y[t]==g].var(axis=0)
    for q,feature in enumerate(FEATURES):
        varrows.append(dict(genre=g,feature=feature,variance=original_variance[q],
                            epsilon=scaled.epsilon_,floor_ratio=scaled.epsilon_/max(original_variance[q],1e-300)))
table('nb_within_class_variances',varrows)
# save implementation
(OUT/'GaussianNB_fit_source.txt').write_text(inspect.getsource(GaussianNB._partial_fit),encoding='utf-8')

## 3.2 Part 2: assumptions and confidence with duplicated features

In [ ]:
assumptions=[]; corrrows=[]
fig,axs=plt.subplots(1,2,figsize=(11,4))
for g in ['sleep','techno','comedy']:
    sub=X[y==g]; q=FEATURES.index('instrumentalness'); en=FEATURES.index('energy'); loud=FEATURES.index('loudness')
    assumptions.append(dict(genre=g,n=len(sub),feature='instrumentalness',skew=skew(sub[:,q]),
        zero_fraction=np.mean(sub[:,q]==0),below_001_fraction=np.mean(sub[:,q]<.01)))
    corrrows.append(dict(genre=g,feature_1='energy',feature_2='loudness',pearson_r=np.corrcoef(sub[:,en],sub[:,loud])[0,1]))
    axs[0].hist(sub[:,q],bins=25,alpha=.4,density=True,label=g)
    axs[1].scatter(sub[:,en],sub[:,loud],s=8,alpha=.4,label=g)
axs[0].set(xlabel='instrumentalness',ylabel='density'); axs[1].set(xlabel='energy',ylabel='loudness');
for ax in axs: ax.legend()
savefig('nb_conditional_assumptions'); table('nb_conditional_gaussian_evidence',assumptions); table('nb_conditional_correlations',corrrows)
confidence=[]; bins=[]
fig,ax=plt.subplots(figsize=(6,5)); ax.plot([0,1],[0,1],'k--')
for copies in [0,1,3,5]:
    xx=np.c_[X,np.repeat(X[:,[FEATURES.index('energy')]],copies,axis=1)]
    _,p,probs=evaluate(make_pipeline(StandardScaler(),GaussianNB(var_smoothing=NB_SMOOTH)),f'nb_copies_{copies}',xx=xx)
    conf=probs.max(axis=1); correct=(p==y)
    confidence.append(dict(extra_copies=copies,accuracy=correct.mean(),mean_confidence=conf.mean(),
        macro_f1=f1_score(y,p,average='macro'),brier_multiclass=np.mean(np.sum((probs-label_binarize(y,classes=LABELS))**2,axis=1))))
    assigned=np.minimum((conf*10).astype(int),9); xb=[]; yb=[]
    for j in range(10):
        mask=assigned==j
        row=dict(extra_copies=copies,bin=j,count=int(mask.sum()),confidence=conf[mask].mean() if mask.any() else np.nan,
                 accuracy=correct[mask].mean() if mask.any() else np.nan)
        bins.append(row)
        if mask.any(): xb.append(row['confidence']); yb.append(row['accuracy'])
    ax.plot(xb,yb,'o-',label=f'{copies} extra copies')
ax.set(xlabel='mean maximum posterior',ylabel='empirical accuracy'); ax.legend(); savefig('nb_reliability')
table('nb_confidence',confidence); table('nb_reliability_bins',bins)

## 3.3 Decision trees

In [ ]:
t,v=outer[0]; tree=DecisionTreeClassifier(max_depth=3,random_state=SEED).fit(X[t],y[t])
plt.figure(figsize=(24,10)); plot_tree(tree,feature_names=FEATURES,class_names=tree.classes_,filled=True,fontsize=7,proportion=True); savefig('tree_depth3')
tr=tree.tree_; node_rows=[]
for node in range(tr.node_count):
    left,right=tr.children_left[node],tr.children_right[node]
    if left<0: continue
    n=tr.weighted_n_node_samples
    gain=tr.impurity[node]-(n[left]*tr.impurity[left]+n[right]*tr.impurity[right])/n[node]
    node_rows.append(dict(node=node,feature=FEATURES[tr.feature[node]],threshold=tr.threshold[node],
        samples=n[node],local_gini_decrease=gain,weighted_gini_decrease=n[node]/n[0]*gain,
        left_majority=tree.classes_[tr.value[left].argmax()],right_majority=tree.classes_[tr.value[right].argmax()]))
table('tree_split_details',node_rows)
leaf_classes=set(tree.classes_[tr.value[tr.children_left<0].reshape(-1,len(LABELS)).argmax(axis=1)])
print('Genres absent from shallow-tree leaf predictions:',sorted(set(LABELS)-leaf_classes))
d,p,_=evaluate(DecisionTreeClassifier(max_depth=3,random_state=SEED),'tree_depth3'); table('tree_depth3_scores',d)
print('Genres never predicted in depth-3 OOF predictions:',sorted(set(LABELS)-set(p)))
tree_curves=[]
for parameter,values in [('max_depth',[1,2,3,5,8,12,20,None]),('min_samples_leaf',[1,2,5,10,30,100,300])]:
    rows=[]
    for val in values:
        d,_,_=evaluate(DecisionTreeClassifier(random_state=SEED,**{parameter:val}),f'tree_{parameter}_{val}',save_predictions=False)
        d[parameter]=50 if val is None else val; d['setting']='unlimited' if val is None else str(val); rows.append(d)
    d=table('tree_'+parameter+'_curves',pd.concat(rows)); curve_plot(d,parameter,'tree_'+parameter,True)

## 3.3 Part 2: prediction changes

In [ ]:
ensemble_rows=[]; stability=[]
for name,constructor in [('bagging',lambda n: BaggingClassifier(n_estimators=n,random_state=SEED,n_jobs=1)),
                          ('forest',lambda n: RandomForestClassifier(n_estimators=n,max_features='sqrt',random_state=SEED,n_jobs=1))]:
    for n in [1,5,20,50,100,300]:
        print('Ensemble diagnostic',name,n,flush=True)
        d,_,_=evaluate(constructor(n),name+'_'+str(n),save_predictions=False)
        d['trees']=n; d['family']=name; ensemble_rows.append(d)
ensembles=table('ensemble_curves',pd.concat(ensemble_rows))
for name in ['bagging','forest']: curve_plot(ensembles[ensembles.family==name],'trees',name+'_tree_count',True)
# Paired bootstrap resamples are shared by all families; five pairs assess sampling instability.
t,v=outer[0]; rng=np.random.default_rng(SEED)
boot_pairs=[(rng.choice(t,len(t),replace=True),rng.choice(t,len(t),replace=True)) for _ in range(5)]
for name,model in [('tree',DecisionTreeClassifier(random_state=SEED)),
    ('bagging',BaggingClassifier(n_estimators=300,random_state=SEED,n_jobs=1)),
    ('forest',RandomForestClassifier(n_estimators=300,random_state=SEED,n_jobs=1))]:
    for pair,(b1,b2) in enumerate(boot_pairs):
        p1=clone(model).fit(X[b1],y[b1]).predict(X[v]); p2=clone(model).fit(X[b2],y[b2]).predict(X[v])
        stability.append(dict(model=name,pair=pair+1,disagreement=np.mean(p1!=p2),
            accuracy_1=accuracy_score(y[v],p1),accuracy_2=accuracy_score(y[v],p2)))
table('bootstrap_prediction_instability',stability)

## 3.4 SVM

In [ ]:
CS=[.1,1,10,100,1000]; GAMMAS=[.001,.01,.1,1,10]
svm_rows=[]
for c in CS:
    for gamma in GAMMAS:
        print('SVM diagnostic',c,gamma,flush=True)
        d,_,_=evaluate(make_pipeline(StandardScaler(),SVC(C=c,gamma=gamma,cache_size=512)),f'svm_{c}_{gamma}',save_predictions=False)
        d['C']=c; d['gamma']=gamma; svm_rows.append(d)
svm_curves=table('svm_grid',pd.concat(svm_rows))
fig,axs=plt.subplots(2,2,figsize=(11,9))
for ax,col in zip(axs.flat,['train_accuracy','accuracy','train_macro_f1','macro_f1']):
    pivot=svm_curves.pivot_table(index='C',columns='gamma',values=col)
    im=ax.imshow(pivot,origin='lower',vmin=0,vmax=1,aspect='auto')
    ax.set(xticks=range(5),xticklabels=GAMMAS,yticks=range(5),yticklabels=CS,xlabel='gamma',ylabel='C',title=col)
    for i in range(5):
        for j in range(5): ax.text(j,i,f'{pivot.iloc[i,j]:.2f}',ha='center',va='center',fontsize=8,color='white' if pivot.iloc[i,j]<.5 else 'black')
fig.colorbar(im,ax=axs.ravel().tolist()); plt.savefig(OUT/'svm_heatmaps.png',dpi=160,bbox_inches='tight'); plt.show(); plt.close()
svrows=[]
for c in [.1,1,10,100,1000]:
    m=make_pipeline(StandardScaler(),SVC(C=c,gamma=.01,cache_size=512)).fit(X,y)
    svc=m[-1]; support_labels=y[svc.support_]
    _,pred,_=evaluate(make_pipeline(StandardScaler(),SVC(C=c,gamma=.01,cache_size=512)),f'sv_C_{c}',save_predictions=False)
    ct=class_table(y,pred).set_index('genre')
    for g in LABELS: svrows.append(dict(C=c,gamma=.01,genre=g,training_count=sum(y==g),
        support_count=sum(support_labels==g),support_fraction=np.mean(np.isin(np.flatnonzero(y==g),svc.support_)),oof_f1=ct.loc[g,'f1']))
svdf=table('svm_support_vectors',svrows)
fig,axs=plt.subplots(1,2,figsize=(11,4))
for c,sub in svdf.groupby('C'):
    axs[0].scatter(sub.training_count,sub.support_fraction,label=f'C={c}')
    axs[1].scatter(sub.oof_f1,sub.support_fraction,label=f'C={c}')
axs[0].set(xlabel='genre training count',ylabel='support-vector fraction'); axs[1].set(xlabel='OOF genre F1',ylabel='support-vector fraction'); axs[1].legend(); savefig('svm_support_relationships')

## 4.1 Conclusions from the experiment

In [ ]:
# Every outer validation fold is excluded from both preprocessing and inner selection.
specs={
 'knn':(make_pipeline(StandardScaler(),KNeighborsClassifier(n_jobs=1)),{'kneighborsclassifier__n_neighbors':KS}),
 'bayes':(make_pipeline(StandardScaler(),GaussianNB()),{'gaussiannb__var_smoothing':SMOOTH}),
 'tree':(DecisionTreeClassifier(random_state=SEED),{'max_depth':[3,8,20,None],'min_samples_leaf':[1,5,30]}),
 'bagging':(BaggingClassifier(random_state=SEED,n_jobs=1),{'n_estimators':[50,100,300]}),
 'forest':(RandomForestClassifier(random_state=SEED,n_jobs=1),{'n_estimators':[50,100,300]}),
 'svm':(make_pipeline(StandardScaler(),SVC(cache_size=512)),{'svc__C':CS,'svc__gamma':GAMMAS})}
nested=[]; fitted={}; selected={}; nested_predictions={}
for name,(model,grid) in specs.items():
    pred=np.empty_like(y)
    for f,(t,v) in enumerate(outer):
        print('Nested CV',name,'outer fold',f+1,flush=True)
        search=GridSearchCV(model,grid,scoring='f1_macro',cv=inner_cv(),n_jobs=1,refit=True,error_score='raise').fit(X[t],y[t])
        pred[v]=search.predict(X[v]); nested.append(dict(model=name,fold=f+1,**scores(y[v],pred[v]),parameters=json.dumps(search.best_params_)))
    nested_predictions[name]=pred
    table(name+'_nested_per_class',class_table(y,pred))
    pd.DataFrame(dict(original_row=dev,truth=y,prediction=pred)).to_csv(OUT/(name+'_nested_oof.csv'),index=False)
    final=GridSearchCV(model,grid,scoring='f1_macro',cv=inner_cv(),n_jobs=1,refit=True,error_score='raise').fit(X,y)
    fitted[name]=final.best_estimator_; selected[name]=final.best_params_
nested_df=table('nested_fold_scores',nested)
comparison=nested_df.groupby('model')[['accuracy','macro_f1']].agg(['mean','std'])
comparison.columns=['_'.join(c) for c in comparison.columns]; comparison=comparison.reset_index()
comparison['accuracy_minus_macro_f1']=comparison.accuracy_mean-comparison.macro_f1_mean
table('nested_family_comparison',comparison)
BEST=comparison.sort_values(['macro_f1_mean','model'],ascending=[False,True]).iloc[0]['model']
print('Winning family by nested mean macro-F1:',BEST)
(OUT/'locked_multiclass_choices.json').write_text(json.dumps(dict(winner=BEST,parameters=selected),indent=2))

## 4.2 Hip-hop detector

In [ ]:
yb=(y=='hip-hop').astype(int)
binary_specs={
 'binary_bayes':(make_pipeline(StandardScaler(),GaussianNB()),{'gaussiannb__var_smoothing':SMOOTH}),
 'binary_svm':(make_pipeline(StandardScaler(),SVC(cache_size=512)),{'svc__C':[.1,1,10,100], 'svc__gamma':[.001,.01,.1,1]})}
binary_models={}; binary_rows=[]; binary_oof={}
for name,(model,grid) in binary_specs.items():
    oof=np.zeros(len(y)); rows=[]
    for f,(t,v) in enumerate(outer):
        print('Binary nested CV',name,f+1,flush=True)
        search=GridSearchCV(model,grid,scoring='average_precision',cv=inner_cv(),n_jobs=1,error_score='raise').fit(X[t],yb[t])
        m=search.best_estimator_; s=m.decision_function(X[v]) if hasattr(m,'decision_function') else m.predict_proba(X[v])[:,1]
        oof[v]=s; rows.append(dict(model=name,fold=f+1,average_precision=average_precision_score(yb[v],s),roc_auc=roc_auc_score(yb[v],s),parameters=json.dumps(search.best_params_)))
    table(name+'_nested_scores',rows)
    search=GridSearchCV(model,grid,scoring='average_precision',cv=inner_cv(),n_jobs=1,error_score='raise').fit(X,yb)
    binary_models[name]=search.best_estimator_; binary_oof[name]=oof
    pd.DataFrame(dict(original_row=dev,truth=yb,score=oof)).to_csv(OUT/(name+'_nested_oof.csv'),index=False)
    binary_rows.append(dict(model=name,parameters=search.best_params_,oof_average_precision=average_precision_score(yb,oof)))
DETECTOR=max(binary_rows,key=lambda r:r['oof_average_precision'])['model']
# Highest threshold retaining at least ceil(.8*n_positive) positive training OOF scores.
positive_scores=np.sort(binary_oof[DETECTOR][yb==1])[::-1]
THRESHOLD=float(positive_scores[int(np.ceil(.8*len(positive_scores)))-1])
assert recall_score(yb,binary_oof[DETECTOR]>=THRESHOLD)>=.8
(OUT/'locked_binary_choices.json').write_text(json.dumps(dict(detector=DETECTOR,threshold=THRESHOLD,choices=binary_rows),indent=2))
print('Locked curator threshold:',DETECTOR,THRESHOLD)

## 4. Evaluation

In [ ]:
Xt=df.iloc[test][FEATURES].to_numpy(); yt=df.iloc[test].track_genre.to_numpy(); ybt=(yt=='hip-hop').astype(int)
winner=fitted[BEST]
if BEST=='svm':
    winner=clone(winner).set_params(svc__probability=True,svc__random_state=SEED).fit(X,y)
pred=winner.predict(Xt); probabilities=winner.predict_proba(Xt)
assert np.array_equal(winner.classes_,LABELS)
table('test_multiclass_metrics',[dict(model=BEST,**scores(yt,pred))])
ct=class_table(yt,pred)
ct['ovr_roc_auc']=[roc_auc_score(yt==g,probabilities[:,j]) for j,g in enumerate(LABELS)]
ct['most_confused_with']=[pd.Series(pred[(yt==g)&(pred!=g)]).value_counts().index[0] if np.any((yt==g)&(pred!=g)) else '' for g in LABELS]
table('test_multiclass_per_genre',ct)
rare=ct.set_index('genre').loc[RARE].reset_index(); table('test_rare_genres',rare)
table('test_rare_average',[dict(precision=rare.precision.mean(),recall=rare.recall.mean(),f1=rare.f1.mean())])
cm=confusion_matrix(yt,pred,labels=LABELS)
table('test_multiclass_confusion_counts',pd.DataFrame(cm,columns=LABELS).assign(true_genre=LABELS))
fig,ax=plt.subplots(figsize=(13,12)); ConfusionMatrixDisplay(cm,display_labels=LABELS).plot(ax=ax,xticks_rotation=90,colorbar=False,values_format='d',cmap='Blues'); savefig('test_multiclass_confusion')
pd.DataFrame(dict(original_row=test,truth=yt,prediction=pred,**{'prob_'+g:probabilities[:,j] for j,g in enumerate(LABELS)})).to_csv(OUT/'test_multiclass_predictions.csv',index=False)
def binary_metrics(truth,prediction):
    tn,fp,fn,tp=confusion_matrix(truth,prediction,labels=[0,1]).ravel()
    def ratio(a,b): return a/b if b else 0.
    return dict(TN=int(tn),FP=int(fp),FN=int(fn),TP=int(tp),accuracy=ratio(tn+tp,len(truth)),
        TPR=ratio(tp,tp+fn),TNR=ratio(tn,tn+fp),PPV=ratio(tp,tp+fp),NPV=ratio(tn,tn+fn),F1=ratio(2*tp,2*tp+fp+fn))
rows=[dict(model='never_hip_hop',threshold=np.nan,**binary_metrics(ybt,np.zeros(len(ybt),dtype=int)))]; test_scores={}
fig,axs=plt.subplots(1,2,figsize=(11,5))
for name,m in binary_models.items():
    s=m.decision_function(Xt) if hasattr(m,'decision_function') else m.predict_proba(Xt)[:,1]
    default=0 if hasattr(m,'decision_function') else .5; test_scores[name]=s
    fpr,tpr,_=roc_curve(ybt,s); precision,recall,_=precision_recall_curve(ybt,s)
    rows.append(dict(model=name,threshold=default,**binary_metrics(ybt,s>=default),
                     roc_auc=roc_auc_score(ybt,s),pr_auc_trapezoid=auc(recall,precision),average_precision=average_precision_score(ybt,s)))
    axs[0].plot(fpr,tpr,label=name); axs[1].plot(recall,precision,label=name)
    pd.DataFrame(dict(original_row=test,truth=ybt,score=s,prediction=s>=default)).to_csv(OUT/(name+'_test_predictions.csv'),index=False)
axs[0].plot([0,1],[0,1],'k--'); axs[1].axhline(ybt.mean(),color='k',ls='--',label='test prevalence')
axs[0].set(xlabel='false positive rate',ylabel='true positive rate'); axs[1].set(xlabel='recall',ylabel='precision')
for ax in axs: ax.legend()
savefig('binary_roc_pr'); table('binary_test_metrics',rows)
curator=test_scores[DETECTOR]>=THRESHOLD; cost=binary_metrics(ybt,curator)
cost.update(model=DETECTOR,threshold=THRESHOLD,recall_target_met=bool(cost['TPR']>=.8),
    flagged_count=int(sum(curator)),listens_per_1000=float(1000*curator.mean()),
    missed_hip_hop_per_1000=1000*cost['FN']/len(ybt),false_flags_per_1000=1000*cost['FP']/len(ybt),
    training_oof_recall=recall_score(yb,binary_oof[DETECTOR]>=THRESHOLD))
table('curator_test_cost',[cost])
print('Completed. All selections preceded held-out evaluation. Tables and figures:',OUT.resolve())

## Experiment to increase recall


In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import confusion_matrix, recall_score, average_precision_score
from scipy.stats import beta
from threadpoolctl import threadpool_limits
from IPython.display import display
threadpool_limits(1)
SEED=42
OUT=Path('hw2_results')
FEATURES=['danceability','energy','speechiness','acousticness','instrumentalness',
          'liveness','valence','loudness','tempo','key','mode','time_signature','duration_ms']
df=pd.read_csv('processed_tracks.csv')
dev,test=train_test_split(np.arange(len(df)),test_size=.2,stratify=df.track_genre,random_state=SEED)
X=df.iloc[dev][FEATURES].to_numpy(); yb=(df.iloc[dev].track_genre=='hip-hop').to_numpy().astype(int)
params=json.loads((OUT/'locked_binary_choices.json').read_text())['choices']
params=next(r['parameters'] for r in params if r['model']=='binary_svm')
C=params['svc__C']; gamma=params['svc__gamma']
def calibrated_detector():
    # Scaling belongs inside the estimator: each calibration subfold fits its own scaler.
    return CalibratedClassifierCV(
        make_pipeline(StandardScaler(),SVC(C=C,gamma=gamma,cache_size=512)),
        method='sigmoid',cv=StratifiedKFold(3,shuffle=True,random_state=SEED),ensemble=True,n_jobs=1)
cv=list(StratifiedKFold(5,shuffle=True,random_state=SEED).split(X,yb))
oof=np.zeros(len(yb)); fold_diagnostics=[]
for f,(t,v) in enumerate(cv):
    print('Calibrated detector outer fold',f+1,flush=True)
    m=calibrated_detector().fit(X[t],yb[t]); oof[v]=m.predict_proba(X[v])[:,1]
    old=pd.read_csv(OUT/'binary_svm_nested_oof.csv').score.to_numpy()
    fold_diagnostics.append(dict(fold=f+1,n_positive=int(yb[v].sum()),
        original_margin_positive_q20=float(np.quantile(old[v][yb[v]==1],.2)),
        calibrated_positive_q20=float(np.quantile(oof[v][yb[v]==1],.2)),
        average_precision=average_precision_score(yb[v],oof[v])))
pd.DataFrame(fold_diagnostics).to_csv(OUT/'curator_followup_fold_diagnostics.csv',index=False)
pd.DataFrame(dict(original_row=dev,truth=yb,score=oof)).to_csv(OUT/'curator_calibrated_oof.csv',index=False)
# Prespecified for this follow-up, before examining its test predictions: 90% OOF
# recall, adding a 10-percentage-point safety margin over the operational target.
target=.90
positive_scores=np.sort(oof[yb==1])[::-1]
threshold=float(positive_scores[int(np.ceil(target*len(positive_scores)))-1])
tp=int(sum((oof>=threshold)&(yb==1))); n_positive=int(sum(yb))
lower=float(beta.ppf(.05,tp,n_positive-tp+1))
# This binomial bound is descriptive only: cross-fitted scores are dependent,
# and threshold selection uses these same labels. It is not a formal guarantee.
lock=dict(status='exploratory_after_original_test',C=C,gamma=gamma,calibration='sigmoid_3fold_ensemble',
    target_oof_recall=target,threshold=threshold,oof_recall=tp/n_positive,
    descriptive_one_sided_binomial_lower95=lower,
    caveat='OOF models share training data; selected-threshold binomial bound is not a coverage guarantee. Hyperparameters were fixed from original development tuning, not reselected inside follow-up folds.')
(OUT/'locked_curator_followup.json').write_text(json.dumps(lock,indent=2))
# Fit exactly the same calibrated-ensemble procedure on the whole development set.
final=calibrated_detector().fit(X,yb)
Xt=df.iloc[test][FEATURES].to_numpy(); ybt=(df.iloc[test].track_genre=='hip-hop').to_numpy().astype(int)
prob=final.predict_proba(Xt)[:,1]; flags=prob>=threshold
tn,fp,fn,tp=confusion_matrix(ybt,flags,labels=[0,1]).ravel()
result=dict(method='calibrated_svm_oof90_exploratory',threshold=threshold,
    training_oof_recall=lock['oof_recall'],TN=int(tn),FP=int(fp),FN=int(fn),TP=int(tp),
    test_recall=float(tp/(tp+fn)),test_precision=float(tp/(tp+fp)) if tp+fp else 0.,
    recall_target_met=bool(tp/(tp+fn)>=.8),flagged_count=int(flags.sum()),
    listens_per_1000=float(1000*flags.mean()),missed_hip_hop_per_1000=float(1000*fn/len(ybt)),
    interpretation='Post-hoc exploratory follow-up; original holdout is already observed, not fresh validation.')
pd.DataFrame([result]).to_csv(OUT/'curator_followup_test_cost.csv',index=False)
pd.DataFrame(dict(original_row=test,truth=ybt,probability=prob,prediction=flags)).to_csv(OUT/'curator_followup_test_predictions.csv',index=False)
display(pd.DataFrame([lock])); display(pd.DataFrame([result])); print(json.dumps(result,indent=2))
# Verify from persisted predictions rather than estimator internals.
p=pd.read_csv(OUT/'curator_followup_test_predictions.csv')
assert np.isclose(recall_score(p.truth,p.probability>=threshold),result['test_recall'])
assert np.isclose(1000*np.mean(p.probability>=threshold),result['listens_per_1000'])
assert recall_score(yb,oof>=threshold)>=target
assert not set(dev)&set(test)
print('Follow-up prediction, cost, OOF target, and split checks passed.')


## Produce figures


In [ ]:
from pathlib import Path
import html, json
import pandas as pd
from IPython.display import SVG, display
ROOT=Path.cwd()
OUT=ROOT/'hw2_results'
ASSETS=ROOT/'report_visual_assets'
ASSETS.mkdir(exist_ok=True)
def csv(name): return pd.read_csv(OUT/(name+'.csv'))
BLUE='#2875a7'; ORANGE='#dc7b27'; GRAY='#dce5ec'
def txt(x,y,text,size=20,anchor='start',color='#202a32'):
    return f'<text x="{x}" y="{y}" font-family="Arial" font-size="{size}" text-anchor="{anchor}" fill="{color}">{html.escape(str(text))}</text>'
def rect(x,y,w,h,color): return f'<rect x="{x}" y="{y}" width="{w}" height="{h}" fill="{color}"/>'
def line(x1,y1,x2,y2,color='#ccd5db',dash=''):
    return f'<line x1="{x1}" y1="{y1}" x2="{x2}" y2="{y2}" stroke="{color}" stroke-width="2" stroke-dasharray="{dash}"/>'
def svg(name,w,h,body):
    (ASSETS/(name+'.svg')).write_text(f'<svg xmlns="http://www.w3.org/2000/svg" width="{w}" height="{h}" viewBox="0 0 {w} {h}"><rect width="100%" height="100%" fill="white"/>'+''.join(body)+'</svg>',encoding='utf-8')

comp=csv('nested_family_comparison').set_index('model')
names=[('knn','k-NN'),('bayes','Naive Bayes'),('tree','Tree'),('bagging','Bagging'),('forest','Random forest'),('svm','RBF SVM')]
b=[txt(240,28,'Accuracy',22,color=BLUE),txt(440,28,'Macro-F1',22,color=ORANGE)]
for i,(key,label) in enumerate(names):
    yy=65+i*58; b.append(txt(205,yy+24,label,21,'end'))
    for j,(metric,color) in enumerate([('accuracy_mean',BLUE),('macro_f1_mean',ORANGE)]):
        value=comp.loc[key,metric]; b.append(rect(230,yy+j*23,650*value,18,color)); b.append(txt(240+650*value,yy+16+j*23,f'{value:.3f}',17))
svg('model_comparison',1000,430,b)
terms=csv('nb_log_odds_terms'); b=[txt(30,28,'Positive terms favor techno     Negative terms favor grunge',20)]
for i,r in enumerate(terms.itertuples()):
    yy=51+i*31; xx=510+180*r.value; b.append(txt(255,yy+16,r.term,20,'end')); b.append(rect(min(510,xx),yy,abs(xx-510),21,BLUE if r.value>=0 else ORANGE)); b.append(txt(xx+(9 if r.value>=0 else -9),yy+17,f'{r.value:+.3f}',18,'start' if r.value>=0 else 'end'))
b.append(line(510,43,510,493,'#778899')); svg('nb_terms',1000,510,b)
ct=csv('test_multiclass_per_genre'); b=[txt(250,25,'F1',22,color=BLUE),txt(450,25,'One versus rest AUC',22,color=ORANGE)]
for i,r in enumerate(ct.itertuples()):
    yy=48+i*35; b.append(txt(198,yy+18,r.genre,19,'end'))
    for j,(value,color) in enumerate([(r.f1,BLUE),(r.ovr_roc_auc,ORANGE)]):
        b.append(rect(220,yy+j*13,650*value,10,color)); b.append(txt(230+650*value,yy+10+j*13,f'{value:.2f}',14))
svg('genre_f1_auc',1000,780,b)
old=csv('curator_test_cost').iloc[0]; new=csv('curator_followup_test_cost').iloc[0]
b=[]
for panel,(title,values,maxval,unit) in enumerate([
    ('Hip-hop recall',[100*old.TPR,100*new.test_recall],100,'%'),
    ('Tracks to listen to per 1000',[old.listens_per_1000,new.listens_per_1000],400,'')]):
    xx=panel*500; b.append(txt(xx+20,30,title,22));
    for j,(label,value,color) in enumerate(zip(['Original','Follow-up'],values,[BLUE,ORANGE])):
        yy=80+j*115; b.append(txt(xx+25,yy,label,21)); b.append(rect(xx+25,yy+14,390*value/maxval,45,color)); b.append(txt(xx+25,yy+83,f'{value:.1f}{unit}',23))
svg('curator_tradeoff',1000,335,b)

spl=csv('tree_split_details').set_index('node')
positions={0:(600,45),1:(320,175),8:(970,175),2:(155,305),5:(480,305),9:(875,305),12:(1090,435),3:(70,435),4:(240,435),6:(410,435),7:(570,435),10:(790,435),11:(950,435)}
edges=[(0,1),(0,8),(1,2),(1,5),(2,3),(2,4),(5,6),(5,7),(8,9),(8,12),(9,10),(9,11)]
b=[]
for parent,child in edges:
    x,y=positions[parent]; xx,yy=positions[child]; b.append(line(x,y+50,xx,yy,'#8a99a5'))
b.extend([txt(425,145,'yes',18),txt(805,145,'no',18)])
for node,(x,y) in positions.items():
    if node in spl.index:
        r=spl.loc[node]; w=260 if node==0 else 240
        b.append(rect(x-w/2,y,w,58,'#e7eff5'))
        b.append(txt(x,y+24,r.feature,21,'middle')); b.append(txt(x,y+47,f'<= {r.threshold:.4f}',21,'middle'))
    else:
        leaf={3:'sleep',4:'ambient',6:'r-n-b',7:'techno',10:'comedy',11:'comedy',12:'r-n-b'}[node]
        b.append(rect(x-68,y,136,45,'#dcebdc')); b.append(txt(x,y+29,leaf,20,'middle'))
svg('tree_readable',1220,510,b)

for figure in ['model_comparison','nb_terms','tree_readable','genre_f1_auc','curator_tradeoff']:
    display(SVG(filename=str(ASSETS/(figure+'.svg'))))

def evidence_table(name,frame):
    frame.to_csv(OUT/(name+'.csv'),index=False)
    display(frame)
full=csv('tree_max_depth_curves').query('max_depth==50')[['accuracy','macro_f1']].mean()
ens=csv('ensemble_curves').query('trees==300').groupby('family')[['accuracy','macro_f1']].mean()
evidence_table('report_full_tree_ensemble_comparison',pd.DataFrame([
    dict(model='fully_grown_tree',**full.to_dict()),
    dict(model='bagging_300',**ens.loc['bagging'].to_dict()),
    dict(model='forest_300',**ens.loc['forest'].to_dict())]))
evidence_table('report_nb_raw_scaled_summary',csv('nb_raw_scaled_scores').groupby('model')[['accuracy','macro_f1']].agg(['mean','std']).reset_index())
